: 

In [ ]:
https://arctic-shift.photon-reddit.com/api/posts/search?author=Its_Khaleeesii_Bitch&before=1743465600000&limit=auto&sort=asc&after=1735689600000&meta-app=download-tool


In [ ]:
import requests
import json
from pathlib import Path
from datetime import datetime

RELEVANT_FIELDS = [
    "author",  # Username of the author
    "author_created_utc",  # Account creation timestamp
    "author_fullname",  # Unique identifier for the author
    "subreddit",  # Subreddit where the post was made
    "num_comments",  # Number of comments on the post
    "title",  # Title of the post
    "permalink",  # URL to the post on Reddit
    "selftext",  # Text content of the post (if applicable)
    "created_utc"  # Timestamp when the post was created
]

authors = [
    "Its_Khaleeesii_Bitch",
    "crackerthatcantspell",
    "Affectionate_Low4990",
    "Alternative-Yak1048",
    "tonyfo98",
    "FunMoneyLife",
    "Important_Berry9732",
    "Odd-Nobody9097",
    "bet055",
    "tsb9876"
]

def to_millis(dt_str):
    """Convert 'YYYY-MM-DD' string to milliseconds since epoch (UTC)."""
    return int(datetime.strptime(dt_str, "%Y-%m-%d").timestamp() * 1000)

def download_and_filter_user_data(username, after_str, before_str, out_path=None):
    """
    Download and filter Reddit posts from Arctic-Shift API for a given user and date range, with pagination.
    """
    BASE_URL = "https://arctic-shift.photon-reddit.com/api/posts/search"
    after = to_millis(after_str)
    print(after)
    before = to_millis(before_str)
    print(before)

    if out_path is None:
        out_path = Path(f"{username}_posts_{after_str}_to_{before_str}_filtered.jsonl")
    else:
        out_path = Path(out_path)

    CHUNK = 1 << 14  # 16 kB
    count_total = 0

    print(f"➜ Downloading and filtering posts for user '{username}' to {out_path.resolve()}")

    with out_path.open("w", encoding="utf-8") as outfile:
        while True:
            params = {
                "author": username,  # Fetch data for the specific user
                "after": after,
                "before": before,
                "limit": "auto",
                "sort": "asc",
                "meta-app": "download-tool"
            }
            with requests.get(BASE_URL, params=params, stream=True, timeout=120) as r:
                r.raise_for_status()
                data_buffer = b""
                for chunk in r.iter_content(CHUNK):
                    if chunk:
                        data_buffer += chunk
                try:
                    decoded = data_buffer.decode('utf-8')
                    posts_json = json.loads(decoded)
                    posts = posts_json["data"]
                except Exception as e:
                    print("Error parsing response:", e)
                    print("Raw data (first 500 chars):", decoded[:500])
                    break

                if not posts:
                    print("No more posts found; exiting loop.")
                    break  # No more data, exit

                for post in posts:
                    filtered = {key: post[key] for key in RELEVANT_FIELDS if key in post}
                    outfile.write(json.dumps(filtered) + '\n')
                count_total += len(posts)
                print(f"  ...downloaded {len(posts)} (total so far: {count_total})")

                # If fewer than 1000 results, this is the last batch
                if len(posts) < 1000:
                    print("Last batch received; exiting loop.")
                    break

                # Next after: max created_utc + 1 (to avoid duplicates)
                max_created_utc = max(post["created_utc"] for post in posts)
                print(max_created_utc)
                # If max_created_utc >= before, we're done (avoid invalid range)
                if max_created_utc >= before:
                    print("Reached or exceeded 'before' timestamp; exiting loop.")
                    break
                after = max_created_utc + 1
                print(after)


    print(f"✅ Done! Total posts saved: {count_total}. File: {out_path}")

# Example usage


def main():
    download_and_filter_user_data("Its_Khaleeesii_Bitch", "2025-01-01", "2023-04-31")

if __name__ == "__main__":
    main()


: 